# Notebook 01B — Multi-Noise Data Generation

**What this notebook does differently from 01:**  
Notebook 01 trained on p=0.001 only — causing both decoders to collapse at p≥0.01.  
This notebook generates a **mixed-noise training set** by sampling equal batches at every noise level.  
The model then sees the full noise range during training, producing a decoder that generalises.

**Strategy:**
```
p ∈ {0.0005, 0.001, 0.002, 0.005, 0.01, 0.02}  ← 6 noise levels
16,667 shots per level  × 6 levels  = 100,000 total training shots
Sweep data: 50,000 shots per level (same as notebook 01 — for fair comparison)
```

**Prerequisite:** None. Run this independently of notebook 01.
```bash
pip install stim numpy
```

In [1]:
import numpy as np
import stim
import os

SEED     = 42
DISTANCE = 3
ROUNDS   = 9
np.random.seed(SEED)

print(f"Stim  : {stim.__version__}")
print(f"NumPy : {np.__version__}")

Stim  : 1.15.0
NumPy : 2.2.6


---
## 1. Circuit Builder

In [2]:
def build_circuit(p, distance=DISTANCE, rounds=ROUNDS):
    """Build rotated surface code circuit at noise level p."""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        rounds=rounds,
        distance=distance,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )


def sample_circuit(circuit, num_shots):
    """
    Sample from circuit. Returns:
      det  : (N, num_detectors)  float32  detection events
      raw  : (N, num_detectors)  float32  raw syndrome measurements (cols 0..71)
      obs  : (N,)                float32  observable flips
    """
    sampler    = circuit.compile_sampler()
    raw_all    = np.array(sampler.sample(num_shots), dtype=np.bool_)

    m2d        = circuit.compile_m2d_converter()
    det_events, obs_flips = m2d.convert(
        measurements=raw_all, separate_observables=True)

    num_det    = det_events.shape[1]          # 72 for d=3 r=9
    raw_syndrome = raw_all[:, :num_det]        # exclude final data qubit readout

    return (det_events.astype(np.float32),
            raw_syndrome.astype(np.float32),
            obs_flips.squeeze().astype(np.float32))


# Verify on one circuit
circ_test = build_circuit(0.001)
print(f"Circuit qubits       : {circ_test.num_qubits}")
print(f"Circuit detectors    : {circ_test.num_detectors}")
print(f"Circuit observables  : {circ_test.num_observables}")
det_t, raw_t, obs_t = sample_circuit(circ_test, 1000)
print(f"\nSample shapes:")
print(f"  det : {det_t.shape}")
print(f"  raw : {raw_t.shape}")
print(f"  obs : {obs_t.shape}")

Circuit qubits       : 26
Circuit detectors    : 72
Circuit observables  : 1

Sample shapes:
  det : (1000, 72)
  raw : (1000, 72)
  obs : (1000,)


---
## 2. Mixed-Noise Training Set

Each noise level contributes an equal number of shots.  
The dataset is **shuffled** so noise levels are interleaved — the model cannot exploit ordering.

In [3]:
NOISE_LEVELS     = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]
TOTAL_TRAIN_SHOTS = 100_000
SHOTS_PER_LEVEL  = TOTAL_TRAIN_SHOTS // len(NOISE_LEVELS)   # 16,667 each

print(f"Mixed-noise training set")
print(f"  Noise levels  : {NOISE_LEVELS}")
print(f"  Shots/level   : {SHOTS_PER_LEVEL:,}")
print(f"  Total shots   : {SHOTS_PER_LEVEL * len(NOISE_LEVELS):,}")
print()

det_parts, raw_parts, obs_parts = [], [], []

print(f"{'p':>8}  {'Shots':>8}  {'LER':>8}")
print("-" * 32)

for p in NOISE_LEVELS:
    circ       = build_circuit(p)
    det, raw, obs = sample_circuit(circ, SHOTS_PER_LEVEL)
    det_parts.append(det)
    raw_parts.append(raw)
    obs_parts.append(obs)
    print(f"  p={p:.4f}  {SHOTS_PER_LEVEL:>8,}  {obs.mean():.4f}")

# Concatenate all levels
det_mixed = np.concatenate(det_parts, axis=0)
raw_mixed = np.concatenate(raw_parts, axis=0)
obs_mixed = np.concatenate(obs_parts, axis=0)

# Shuffle so levels are interleaved
rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(obs_mixed))
det_mixed = det_mixed[perm]
raw_mixed = raw_mixed[perm]
obs_mixed = obs_mixed[perm]

print()
print(f"Mixed training set (after shuffle):")
print(f"  det_mixed : {det_mixed.shape}")
print(f"  raw_mixed : {raw_mixed.shape}")
print(f"  obs_mixed : {obs_mixed.shape}")
print(f"  Overall LER : {obs_mixed.mean():.4f}")

Mixed-noise training set
  Noise levels  : [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]
  Shots/level   : 16,666
  Total shots   : 99,996

       p     Shots       LER
--------------------------------
  p=0.0005    16,666  0.0287
  p=0.0010    16,666  0.0574
  p=0.0020    16,666  0.1000
  p=0.0050    16,666  0.2189
  p=0.0100    16,666  0.3466
  p=0.0200    16,666  0.4540

Mixed training set (after shuffle):
  det_mixed : (99996, 72)
  raw_mixed : (99996, 72)
  obs_mixed : (99996,)
  Overall LER : 0.2009


---
## 3. Save Mixed Training Data

In [4]:
os.makedirs("data/mixed", exist_ok=True)

np.save("data/mixed/detection_events.npy", det_mixed)
np.save("data/mixed/raw_measurements.npy", raw_mixed)
np.save("data/mixed/observable_flips.npy",  obs_mixed)

print("Saved mixed training data to data/mixed/")
print(f"  detection_events.npy  {det_mixed.shape}  {det_mixed.nbytes/1e6:.1f} MB")
print(f"  raw_measurements.npy  {raw_mixed.shape}  {raw_mixed.nbytes/1e6:.1f} MB")
print(f"  observable_flips.npy  {obs_mixed.shape}")

Saved mixed training data to data/mixed/
  detection_events.npy  (99996, 72)  28.8 MB
  raw_measurements.npy  (99996, 72)  28.8 MB
  observable_flips.npy  (99996,)


---
## 4. Sweep Data (same as notebook 01 — for fair comparison)

The sweep data is generated fresh here at 50,000 shots per level.  
Saved to `data/sweep/` — same location notebook 03B will look.

In [5]:
SWEEP_SHOTS = 50_000
os.makedirs("data/sweep", exist_ok=True)

print(f"Sweep data ({SWEEP_SHOTS:,} shots per level)")
print(f"{'p':>8}  {'LER':>8}")
print("-" * 20)

for p in NOISE_LEVELS:
    tag  = f"{int(p * 10000):04d}"
    circ = build_circuit(p)
    det, raw, obs = sample_circuit(circ, SWEEP_SHOTS)
    np.save(f"data/sweep/det_p{tag}.npy", det)
    np.save(f"data/sweep/raw_p{tag}.npy", raw)
    np.save(f"data/sweep/obs_p{tag}.npy", obs)
    print(f"  p={p:.4f}  LER={obs.mean():.5f}  saved → sweep/*_p{tag}.npy")

print("\nDone. Notebook 02B and 03B are ready to run.")

Sweep data (50,000 shots per level)
       p       LER
--------------------
  p=0.0005  LER=0.02794  saved → sweep/*_p0005.npy
  p=0.0010  LER=0.05446  saved → sweep/*_p0010.npy
  p=0.0020  LER=0.10400  saved → sweep/*_p0020.npy
  p=0.0050  LER=0.21678  saved → sweep/*_p0050.npy
  p=0.0100  LER=0.34436  saved → sweep/*_p0100.npy
  p=0.0200  LER=0.45288  saved → sweep/*_p0200.npy

Done. Notebook 02B and 03B are ready to run.
